This notebook introduces a minimal attention_based neural language learner trained on an artificial language with singular and plural number particles.

The experiment investigates whether the learner can generalise a distributional regularity to a noun-number combination absent from its training data.

Step 1: Define examples that belong to our artificial language

In [1825]:
examples = ["one dax pa", "two dax ti"]

In [1826]:
print(examples)

['one dax pa', 'two dax ti']


Step 2: Add examples of artificial language to the training data

In [1827]:
examples.extend(["one mit pa", "two mit ti"])

In [1828]:
print(examples)

['one dax pa', 'two dax ti', 'one mit pa', 'two mit ti']


Step 3: Add an example containing a new noun (singular only) to test generalisation later

In [1829]:
examples.append("one wug pa")

In [1830]:
print(examples)

['one dax pa', 'two dax ti', 'one mit pa', 'two mit ti', 'one wug pa']


Step 4: Count examples in the training data

In [1831]:
len(examples)

5

Step 5: Split one example into individual words (tokens)

In [1832]:
examples[1].split()

['two', 'dax', 'ti']

Step 6: Tokenize all training examples and display the result

In [1833]:
for sentence in examples:
  print(sentence.split())

['one', 'dax', 'pa']
['two', 'dax', 'ti']
['one', 'mit', 'pa']
['two', 'mit', 'ti']
['one', 'wug', 'pa']


Step 7: Store the tokenized examples

In [1834]:
tokens = []
for sentence in examples:
  tokens.append(sentence.split())

In [1835]:
print(tokens)

[['one', 'dax', 'pa'], ['two', 'dax', 'ti'], ['one', 'mit', 'pa'], ['two', 'mit', 'ti'], ['one', 'wug', 'pa']]


Step 8: Collect all tokens into a single list (words)

In [1836]:
words = []

for sentence in tokens:
  words.extend(sentence)

In [1837]:
print(words)

['one', 'dax', 'pa', 'two', 'dax', 'ti', 'one', 'mit', 'pa', 'two', 'mit', 'ti', 'one', 'wug', 'pa']


Step 9: Create our vocabulary (unique tokens)


In [1838]:
vocabulary = set(words)

In [1839]:
print(vocabulary)

{'one', 'dax', 'wug', 'mit', 'two', 'pa', 'ti'}


In [1840]:
vocabulary = sorted(vocabulary)

In [1841]:
print(vocabulary)

['dax', 'mit', 'one', 'pa', 'ti', 'two', 'wug']


Step 10: Assign a unique numerical ID to each token


In [1842]:
for number, word in enumerate(vocabulary):
  print(number, word)

0 dax
1 mit
2 one
3 pa
4 ti
5 two
6 wug


In [1843]:
word_to_number = {}

for number, word in enumerate(vocabulary):
  word_to_number[word] = number

In [1844]:
print(word_to_number)

{'dax': 0, 'mit': 1, 'one': 2, 'pa': 3, 'ti': 4, 'two': 5, 'wug': 6}


Step 11: Encode our training examples as sequences of token IDs

In [1845]:
numbers = []

for word in tokens[0]:
  numbers.append(word_to_number[word])

In [1846]:
print(numbers)

[2, 0, 3]


In [1847]:
numerical_examples = []

for sentence in tokens:
  numbers = []
  for word in sentence:
    numbers.append(word_to_number[word])
  numerical_examples.append(numbers)

In [1848]:
print(numerical_examples)

[[2, 0, 3], [5, 0, 4], [2, 1, 3], [5, 1, 4], [2, 6, 3]]


Step 12: Import PyTorch nad convert numerical examples into tensors

In [1849]:
import torch
torch.manual_seed(42)

In [1850]:
first_sentence = torch.tensor(numerical_examples[0])

In [1851]:
print(first_sentence)

tensor([2, 0, 3])


In [1852]:
tensors = []

for numbers in numerical_examples:
  tensors.append(torch.tensor(numbers))

In [1853]:
print(tensors)

[tensor([2, 0, 3]), tensor([5, 0, 4]), tensor([2, 1, 3]), tensor([5, 1, 4]), tensor([2, 6, 3])]


Step 13: Create word embeddings (4 embedding dimensions per token)

In [1854]:
embedding = torch.nn.Embedding(len(vocabulary), 4)

In [1855]:
print(embedding(tensors[0]))

tensor([[-0.7521,  1.6487, -0.3925, -1.4036],
        [ 1.9269,  1.4873,  0.9007, -2.1055],
        [-1.1109,  0.0915, -2.3169, -0.2168]], grad_fn=<EmbeddingBackward0>)


In [1856]:
print(embedding(tensors[0]).shape)

torch.Size([3, 4])


Step 14: Represent word order using positional embeddings

In [1857]:
positions = torch.arange(3)
print(positions)

tensor([0, 1, 2])


In [1858]:
position_embedding = torch.nn.Embedding(3, 4)

In [1859]:
print(position_embedding(positions))

tensor([[-0.3278,  0.7950,  0.2815,  0.0562],
        [ 0.5227, -0.2384, -0.0499,  0.5263],
        [-0.0085,  0.7291,  0.1331,  0.8640]], grad_fn=<EmbeddingBackward0>)


In [1860]:
combined = embedding(tensors[0]) + position_embedding(positions)

In [1861]:
print(combined.shape)

torch.Size([3, 4])


Step 15: Prepare input for attention

In [1862]:
input_batch = combined.unsqueeze(0)

In [1863]:
print(input_batch.shape)

torch.Size([1, 3, 4])


Step 16: Add self-attention mechanism and mask (preventing access to future tokens)

In [1864]:
attention = torch.nn.MultiheadAttention(embed_dim=4, num_heads=1, batch_first=True)

In [1865]:
mask = torch.tensor([
    [False, True, True],
    [False, False, True],
    [False, False, False]
])

In [1866]:
print(mask)

tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])


In [1867]:
output, weights = attention(
    input_batch,
    input_batch,
    input_batch,
    attn_mask=mask
)

In [1868]:
print(output.shape)

torch.Size([1, 3, 4])


In [1869]:
print(weights)

tensor([[[1.0000, 0.0000, 0.0000],
         [0.4388, 0.5612, 0.0000],
         [0.4379, 0.2704, 0.2917]]], grad_fn=<MeanBackward1>)


Step 17: Use attention to predict the next word

In [1870]:
prediction = torch.nn.Linear(4, len(vocabulary))

In [1871]:
scores = prediction(output)

In [1872]:
print(scores.shape)

torch.Size([1, 3, 7])


In [1873]:
print(scores[0, 1, word_to_number["pa"]])
print(scores[0, 1, word_to_number["ti"]])

tensor(0.5728, grad_fn=<SelectBackward0>)
tensor(0.1698, grad_fn=<SelectBackward0>)


In [1874]:
probabilities = torch.softmax(scores[0, 1], dim=0)

In [1875]:
print(probabilities[word_to_number["pa"]])
print(probabilities[word_to_number["ti"]])

tensor(0.2321, grad_fn=<SelectBackward0>)
tensor(0.1552, grad_fn=<SelectBackward0>)


Step 18: Calculate the model's prediction error (loss)

In [1876]:
correct_word = word_to_number["pa"]
loss = -torch.log(probabilities[correct_word])
print(loss.item())

1.460390567779541


In [1877]:
loss.backward() # backpropagation

In [1878]:
parameters = list(embedding.parameters())
parameters += list(position_embedding.parameters())
parameters += list(attention.parameters())
parameters += list(prediction.parameters())

In [1879]:
optimizer = torch.optim.SGD(parameters, lr=0.3)

In [1880]:
optimizer.step()

In [1881]:
combined = embedding(tensors[0]) + position_embedding(positions)

input_batch = combined.unsqueeze(0)

output, weights = attention(input_batch, input_batch, input_batch, attn_mask=mask)

scores = prediction(output)

In [1882]:
probabilities = torch.softmax(scores[0, 1], dim=0)

print(probabilities[word_to_number["pa"]].item())

0.3830033838748932


In [1883]:
new_loss = -torch.log(probabilities[word_to_number["pa"]])

print(new_loss.item())

0.9597114324569702


Step 19: Prepare all examples for training

In [1884]:
for sentence in tensors:
  print(sentence[:2], "->", sentence[2])

tensor([2, 0]) -> tensor(3)
tensor([5, 0]) -> tensor(4)
tensor([2, 1]) -> tensor(3)
tensor([5, 1]) -> tensor(4)
tensor([2, 6]) -> tensor(3)


In [1885]:
for sentence in tensors:
  input_words = sentence[:2]
  combined = embedding(input_words) + position_embedding(positions[:2])
  print(combined.shape)

torch.Size([2, 4])
torch.Size([2, 4])
torch.Size([2, 4])
torch.Size([2, 4])
torch.Size([2, 4])


In [1886]:
mask_2 = torch.tensor([
    [False, True],
    [False, False]
])

In [1887]:
print(mask_2.shape)

torch.Size([2, 2])


In [1888]:
for sentence in tensors:
  input_words = sentence[:2]
  combined = embedding(input_words) + position_embedding(positions[:2])
  input_batch = combined.unsqueeze(0)
  output, weights = attention(input_batch, input_batch, input_batch, attn_mask=mask_2)
  scores = prediction(output)
  print(scores[0, 1].shape)

torch.Size([7])
torch.Size([7])
torch.Size([7])
torch.Size([7])
torch.Size([7])


Step 20: Training the learner

In [1889]:
print(scores[0, 1, word_to_number["pa"]])
print(scores[0, 1, word_to_number["ti"]])

tensor(1.0372, grad_fn=<SelectBackward0>)
tensor(0.6849, grad_fn=<SelectBackward0>)


In [1890]:
for sentence in tensors:
  input_words = sentence[:2]
  combined = embedding(input_words) + position_embedding(positions[:2])
  input_batch = combined.unsqueeze(0)
  output, weights = attention(input_batch, input_batch, input_batch, attn_mask=mask_2)
  scores = prediction(output)
  print(scores[0, 1].shape)

  probabilities = torch.softmax(scores[0, 1], dim=0)
  loss = -torch.log(probabilities[sentence[2]])
  print(loss.item())

torch.Size([7])
0.9597114324569702
torch.Size([7])
2.030026912689209
torch.Size([7])
1.3826470375061035
torch.Size([7])
1.9613709449768066
torch.Size([7])
1.3214764595031738


In [1891]:
for epoch in range(127): # 635 parameter updates (5 examples x 127 epochs)
  total_loss = 0

  for sentence in tensors:
    optimizer.zero_grad()

    input_words = sentence[:2]
    combined = embedding(input_words) + position_embedding(positions[:2])
    input_batch = combined.unsqueeze(0)
    output, weights = attention(input_batch, input_batch, input_batch, attn_mask=mask_2)
    scores = prediction(output)

    probabilities = torch.softmax(scores[0, 1], dim=0)
    loss = -torch.log(probabilities[sentence[2]])

    loss.backward()
    optimizer.step()
    total_loss += loss.item()


  print("Epoch:", epoch + 1, "Average loss:", total_loss / len(tensors))

Epoch: 1 Average loss: 1.573740816116333
Epoch: 2 Average loss: 1.12870591878891
Epoch: 3 Average loss: 0.9202574908733367
Epoch: 4 Average loss: 0.6010895766317844
Epoch: 5 Average loss: 0.37927000131458044
Epoch: 6 Average loss: 0.26554533212911335
Epoch: 7 Average loss: 0.15144232544116676
Epoch: 8 Average loss: 0.07243092671851628
Epoch: 9 Average loss: 0.03372895392822102
Epoch: 10 Average loss: 0.01781260026618838
Epoch: 11 Average loss: 0.010990150926227215
Epoch: 12 Average loss: 0.007585519530402962
Epoch: 13 Average loss: 0.005644650736940093
Epoch: 14 Average loss: 0.004423802332166815
Epoch: 15 Average loss: 0.0035986180351756047
Epoch: 16 Average loss: 0.0030101548654783983
Epoch: 17 Average loss: 0.0025725077888637316
Epoch: 18 Average loss: 0.0022362808711477557
Epoch: 19 Average loss: 0.0019710790831595658
Epoch: 20 Average loss: 0.0017571819760632934
Epoch: 21 Average loss: 0.001581609164713882
Epoch: 22 Average loss: 0.0014352117985254153
Epoch: 23 Average loss: 0.001

Step 21: Testing generalisation

In [1892]:
test_words = torch.tensor([word_to_number["two"], word_to_number["wug"]])

In [1893]:
print(test_words)

tensor([5, 6])


In [1894]:
combined = embedding(test_words) + position_embedding(positions[:2])
input_batch = combined.unsqueeze(0)

In [1895]:
print(input_batch.shape)

torch.Size([1, 2, 4])


In [1896]:
output, weights = attention(input_batch, input_batch, input_batch, attn_mask=mask_2)
scores = prediction(output)

In [1897]:
print(scores.shape)

torch.Size([1, 2, 7])


In [1898]:
probabilities = torch.softmax(scores[0, 1], dim=0)

print("P(pa):", probabilities[word_to_number["pa"]].item())
print("P(ti):", probabilities[word_to_number["ti"]].item())

P(pa): 0.0018424013396725059
P(ti): 0.9915881752967834


* The learner encountered "one dax pa" and "two dax ti"
* It encountered "one mit pa" and "two mit ti"
* It encountered "one wug pa"
* It never encountered two wug ti

After training, where does the model assign its probability following two wug, pa or ti? Is there evidence of generalisation beyond the specific examples in the training data?

Step 22: Final control test (a familiar example)

In [1899]:
test_words = torch.tensor([word_to_number["one"], word_to_number["wug"]])

In [1900]:
combined = embedding(test_words) + position_embedding(positions[:2])

input_batch = combined.unsqueeze(0)

output, weights = attention(input_batch, input_batch, input_batch, attn_mask=mask_2)

scores = prediction(output)

probabilities = torch.softmax(scores[0, 1], dim=0)

print("P(pa):", probabilities[word_to_number["pa"]].item())
print("P(ti):", probabilities[word_to_number["ti"]].item())

P(pa): 0.9998356103897095
P(ti): 3.82644466299098e-05
